# 🦎 Lizard Detection — End-to-End Training Pipeline

**Model:** YOLOv8n (Nano) — single class `lizard`  
**Export:** TFLite FP16 → ready for Android on-device inference  

## Notebook outline
1. Install dependencies  
2. Set up paths & verify dataset  
3. Inspect sample images  
4. Train YOLOv8n  
5. Evaluate (metrics, confusion matrix)  
6. Export → TFLite  
7. Verify TFLite model  
8. Save outputs  

---
> **Kaggle setup:** Add your `lizard_yolo` dataset as an input dataset before running.
> The dataset should be at `/kaggle/input/lizard-yolo/lizard_yolo/`.

## 1 · Install Dependencies

In [ ]:
# Install ultralytics while locking numpy>=2.0 to prevent downgrade.
# The Kaggle torch/torchvision are compiled against numpy 2.x;
# any downgrade to 1.x causes a binary incompatibility crash.
!pip install -q "ultralytics>=8.4.0,<8.5.0" "numpy>=2.0" --no-deps
!pip install -q seaborn

import ultralytics, torch, numpy as np
import tensorflow as tf
print(f"ultralytics : {ultralytics.__version__}")
print(f"torch       : {torch.__version__}")
print(f"numpy       : {np.__version__}")
print(f"tensorflow  : {tf.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
# Test CUDA actually works (torch.cuda.is_available() can be True
# even when the kernel image doesn't match the driver version)
# Kaggle assigned a Tesla P100 (sm_60) but torch 2.10+cu128
# only supports sm_70+. Force CPU to avoid kernel image error.
DEVICE = 'cpu'
print(f'Training device: {DEVICE} (P100/sm_60 incompatible with torch 2.10+cu128)')
print("✅ Environment ready")

In [ ]:
import os, shutil, yaml, json, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from ultralytics import YOLO

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('✅ Imports OK')

## 2 · Paths & Dataset Verification

In [ ]:
# Debug: show what Kaggle actually mounted under /kaggle/input/
import os
print("Contents of /kaggle/input/:")
for item in sorted(os.listdir("/kaggle/input")):
    subpath = f"/kaggle/input/{item}"
    try:
        children = os.listdir(subpath)
        print(f"  {item}/  -> {children[:5]}")
    except:
        print(f"  {item}")


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
KAGGLE_INPUT = Path('/kaggle/input')

# Auto-detect dataset root at any depth under /kaggle/input
DATASET_ROOT = None
for yaml_file in sorted(KAGGLE_INPUT.rglob('dataset.yaml')):
    DATASET_ROOT = yaml_file.parent
    break

assert DATASET_ROOT is not None, (
    f'dataset.yaml not found under {KAGGLE_INPUT}. '
    f'Dirs: {[str(p) for p in KAGGLE_INPUT.rglob("*") if p.is_dir()]}'
)

TRAIN_IMG_DIR = DATASET_ROOT / 'train' / 'images'
VAL_IMG_DIR   = DATASET_ROOT / 'val'   / 'images'

OUTPUT_DIR = Path('/kaggle/working/lizard_detection')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Write patched dataset.yaml to writable /kaggle/working (input dir is read-only)
YAML_PATH = OUTPUT_DIR / 'dataset.yaml'
YAML_PATH.write_text(
    f'path: {DATASET_ROOT}\n'
    f'train: train/images\n'
    f'val:   val/images\n'
    f'\nnc: 1\n'
    f'names: ["Lizard"]\n'
)

n_train = len(list(TRAIN_IMG_DIR.glob('*.jpg')))
n_val   = len(list(VAL_IMG_DIR.glob('*.jpg')))
print(f'Dataset root : {DATASET_ROOT}')
print(f'Train images : {n_train}')
print(f'Val   images : {n_val}')
print(f'YAML path    : {YAML_PATH}')


## 3 · Inspect Sample Images

In [ ]:
def load_yolo_labels(label_path: Path):
    """Return list of (class_id, cx, cy, w, h) tuples from a YOLO .txt file."""
    if not label_path.exists():
        return []
    lines = label_path.read_text().strip().splitlines()
    result = []
    for line in lines:
        parts = line.split()
        if len(parts) == 5:
            result.append((int(parts[0]), *map(float, parts[1:])))
    return result


def show_samples(img_dir: Path, n: int = 8, title: str = 'Sample images'):
    img_files = sorted(img_dir.glob('*.jpg'))[:n]
    label_dir = img_dir.parent.parent / 'labels' / img_dir.parent.name  # …/train/labels/
    # fallback for flat structure
    flat_label_dir = img_dir.parent / 'labels'

    cols = 4
    rows = (len(img_files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

    for ax, img_path in zip(axes, img_files):
        img = Image.open(img_path).convert('RGB')
        w, h = img.size

        # Try to find label file
        lbl_path = label_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            lbl_path = flat_label_dir / (img_path.stem + '.txt')

        ax.imshow(img)
        for cls_id, cx, cy, bw, bh in load_yolo_labels(lbl_path):
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor='#00FF88', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, 'Lizard', color='#00FF88',
                    fontsize=8, fontweight='bold',
                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
        ax.axis('off')
        ax.set_title(img_path.name[:20], fontsize=7)

    for ax in axes[len(img_files):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{title.replace(" ","_")}.png', dpi=120)
    plt.show()
    print(f'Saved preview → {OUTPUT_DIR}')


show_samples(TRAIN_IMG_DIR, n=8, title='Training samples with ground-truth boxes')
show_samples(VAL_IMG_DIR,   n=4, title='Validation samples with ground-truth boxes')

## 4 · Train YOLOv8n

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────
EPOCHS      = 5         # Quick test run
IMG_SIZE    = 416       # 416 balances small-object recall with mobile latency
BATCH_SIZE  = 32        # Adjust down if OOM; 16 works on T4
PATIENCE    = 20        # Early stopping patience (epochs without mAP improvement)
MODEL_BASE  = 'yolov8n.pt'
PROJECT_DIR = str(OUTPUT_DIR / 'runs')
RUN_NAME    = 'lizard_yolov8n'

print(f'Training config:')
print(f'  Base model  : {MODEL_BASE}')
print(f'  Epochs      : {EPOCHS}')
print(f'  Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Early stop  : patience={PATIENCE}')
print(f'  Data yaml   : {YAML_PATH}')

In [ ]:
# ── Load pre-trained YOLOv8n ───────────────────────────────────────────────
model = YOLO(MODEL_BASE)
print(f'Model loaded: {MODEL_BASE}')
print(model.info())

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
results = model.train(
    data        = str(YAML_PATH),
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    patience    = PATIENCE,
    project     = PROJECT_DIR,
    name        = RUN_NAME,
    seed        = SEED,
    workers     = 4,
    # Augmentation — helps with varied lighting & backgrounds in field images
    hsv_h       = 0.015,    # Hue jitter
    hsv_s       = 0.7,      # Saturation jitter
    hsv_v       = 0.4,      # Value/brightness jitter
    degrees     = 10.0,     # Rotation (±10°)
    translate   = 0.1,      # Translation
    scale       = 0.5,      # Scale jitter
    fliplr      = 0.5,      # Horizontal flip
    mosaic      = 1.0,      # Mosaic augmentation (4 images combined)
    mixup       = 0.1,      # MixUp
    # Regularisation
    weight_decay = 0.0005,
    # Logging
    verbose     = True,
    device     = DEVICE,
    plots       = True,
)

print('\n✅ Training complete!')
print(f'Best model saved at: {results.save_dir}')

## 5 · Evaluate — Metrics & Plots

In [ ]:
# ── Locate best weights ────────────────────────────────────────────────────
run_dir   = Path(results.save_dir)
best_pt   = run_dir / 'weights' / 'best.pt'
print(f'Best weights: {best_pt}')

# ── Load best model and validate ──────────────────────────────────────────
best_model = YOLO(str(best_pt))
val_results = best_model.val(
    data   = str(YAML_PATH),
    imgsz  = IMG_SIZE,
    batch  = BATCH_SIZE,
    conf   = 0.25,
    iou    = 0.5,
    plots  = True,
    device = DEVICE,
)

# ── Print key metrics ─────────────────────────────────────────────────────
print('\n' + '='*50)
print('  Validation Metrics (IoU=0.5)')
print('='*50)
print(f'  mAP50    : {val_results.box.map50:.4f}')
print(f'  mAP50-95 : {val_results.box.map:.4f}')
print(f'  Precision: {val_results.box.mp:.4f}')
print(f'  Recall   : {val_results.box.mr:.4f}')
print('='*50)

In [ ]:
# ── Display training curves ────────────────────────────────────────────────
results_csv = run_dir / 'results.csv'
import pandas as pd

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('YOLOv8n Training Curves — Lizard Detection', fontsize=14, fontweight='bold')

    pairs = [
        ('train/box_loss',    'val/box_loss',    'Box Loss'),
        ('train/cls_loss',    'val/cls_loss',    'Class Loss'),
        ('train/dfl_loss',    'val/dfl_loss',    'DFL Loss'),
        ('metrics/precision(B)', None,           'Precision'),
        ('metrics/recall(B)',    None,           'Recall'),
        ('metrics/mAP50(B)',     None,           'mAP@50'),
    ]

    colors = ['#4FC3F7', '#FF8A65']
    for ax, (train_col, val_col, label) in zip(axes.flatten(), pairs):
        if train_col in df.columns:
            ax.plot(df['epoch'], df[train_col], color=colors[0], label='Train', linewidth=2)
        if val_col and val_col in df.columns:
            ax.plot(df['epoch'], df[val_col], color=colors[1], label='Val', linewidth=2, linestyle='--')
        ax.set_title(label)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    curve_path = OUTPUT_DIR / 'training_curves.png'
    plt.savefig(curve_path, dpi=150)
    plt.show()
    print(f'Saved: {curve_path}')
else:
    print('results.csv not found — skipping curves')

In [ ]:
# ── Visualise predictions on validation images ─────────────────────────────
val_images = sorted(VAL_IMG_DIR.glob('*.jpg'))[:6]

cols = 3
rows = (len(val_images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
axes = axes.flatten()

for ax, img_path in zip(axes, val_images):
    preds = best_model.predict(
        source=str(img_path), conf=0.3, iou=0.5, verbose=False, device=DEVICE
    )[0]
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    ax.imshow(img)

    for box in preds.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor='#00E676', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, y1 - 5, f'Lizard {conf:.2f}',
            color='#00E676', fontsize=9, fontweight='bold',
            bbox=dict(facecolor='black', alpha=0.6, pad=2)
        )
    ax.axis('off')

for ax in axes[len(val_images):]:
    ax.axis('off')

fig.suptitle('YOLOv8n Predictions on Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
preds_path = OUTPUT_DIR / 'val_predictions.png'
plt.savefig(preds_path, dpi=150)
plt.show()
print(f'Saved: {preds_path}')

## 6 · Export → TFLite (FP16)

In [ ]:
# Export the best PyTorch model to TFLite with FP16 half-precision
# This produces the smallest model that still runs with hardware acceleration on Android.

print('Exporting to TFLite FP16 …')
export_path = best_model.export(
    format  = 'litert',
    imgsz   = IMG_SIZE,
    half    = False,   # FP16 not supported in litert; use FP32          # FP16 — halves model size vs FP32
    int8    = False,         # Set True for INT8 quantisation (requires calibration data)
    nms     = False,         # Keep NMS off; handle on Android side for flexibility
)

print(f'\n✅ TFLite export complete!')
print(f'   Exported to: {export_path}')

In [ ]:
# Copy the tflite file to working output with a clean name
import glob

tflite_files = glob.glob(str(run_dir / '**' / '*.tflite'), recursive=True)
if not tflite_files:
    tflite_files = glob.glob(str(Path(str(best_pt).replace('.pt','')) + '*.tflite'))

if tflite_files:
    src_tflite = Path(tflite_files[0])
    dst_tflite = OUTPUT_DIR / 'yolov8n_lizard.tflite'
    shutil.copy2(src_tflite, dst_tflite)
    size_mb = dst_tflite.stat().st_size / 1_048_576
    print(f'\nTFLite model copied to: {dst_tflite}')
    print(f'Model size: {size_mb:.2f} MB')
else:
    print('⚠  TFLite file not found via glob — check export_path above')
    print('   Available files in run dir:')
    for f in run_dir.rglob('*'):
        if f.is_file():
            print(f'     {f}')

## 7 · Verify TFLite Model

In [ ]:
import tensorflow as tf

tflite_path = str(OUTPUT_DIR / 'yolov8n_lizard.tflite')

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('='*55)
print('  TFLite Model Tensor Details')
print('='*55)
print('\n  INPUT:')
for d in input_details:
    print(f'    name  : {d["name"]}')
    print(f'    shape : {d["shape"]}')
    print(f'    dtype : {d["dtype"]}')

print('\n  OUTPUT:')
for d in output_details:
    print(f'    name  : {d["name"]}')
    print(f'    shape : {d["shape"]}')
    print(f'    dtype : {d["dtype"]}')

print('='*55)

In [ ]:
# Run a single inference to confirm the model is functional
sample_img_path = sorted(VAL_IMG_DIR.glob('*.jpg'))[0]
sample_img = Image.open(sample_img_path).resize((IMG_SIZE, IMG_SIZE)).convert('RGB')
input_array = np.array(sample_img, dtype=np.float16)[np.newaxis, ...]  # [1, 320, 320, 3]

interpreter.set_tensor(input_details[0]['index'], input_array)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

print(f'✅ TFLite inference OK')
print(f'   Input  image   : {sample_img_path.name}')
print(f'   Output shape   : {output.shape}')
print(f'   Output dtype   : {output.dtype}')
print(f'   Output sample  : {output[0, :, :5]}  (first 5 columns)')

## 8 · Save & Summarise Outputs

In [ ]:
# Copy best.pt alongside tflite for reference
dst_pt = OUTPUT_DIR / 'yolov8n_lizard_best.pt'
shutil.copy2(best_pt, dst_pt)

# Write a metadata JSON for the Android app
metadata = {
    'model_name':     'yolov8n_lizard',
    'input_shape':    [1, IMG_SIZE, IMG_SIZE, 3],
    'input_dtype':    'float16',
    'classes':        ['lizard'],
    'nc':             1,
    'conf_threshold': 0.45,
    'iou_threshold':  0.45,
    'img_size':       IMG_SIZE,
    'mAP50':          round(float(val_results.box.map50), 4),
    'mAP50_95':       round(float(val_results.box.map),   4),
    'precision':      round(float(val_results.box.mp),    4),
    'recall':         round(float(val_results.box.mr),    4),
    'export_format':  'tflite_fp16',
    'notes':          'Trained on Open Images V7 lizard class. Single-stage detection only.'
}

meta_path = OUTPUT_DIR / 'model_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('\n' + '='*55)
print('  Output Files')
print('='*55)
for fpath in sorted(OUTPUT_DIR.iterdir()):
    size = fpath.stat().st_size / 1024
    print(f'  {fpath.name:<40}  {size:>8.1f} KB')

print('\n' + '='*55)
print('  model_metadata.json')
print('='*55)
print(json.dumps(metadata, indent=2))

In [ ]:
# ── Next steps printout ────────────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════╗
║   Training complete! What to do next:               ║
╠══════════════════════════════════════════════════════╣
║  1. Download from /kaggle/working/lizard_detection:  ║
║       • yolov8n_lizard.tflite   → Android assets    ║
║       • model_metadata.json     → Android assets    ║
║       • yolov8n_lizard_best.pt  → keep as backup    ║
║                                                      ║
║  2. In Android Studio:                              ║
║       app/src/main/assets/                          ║
║         └── yolov8n_lizard.tflite                   ║
║         └── model_metadata.json                     ║
╚══════════════════════════════════════════════════════╝
''')